# Лекция: Регрессионный анализ в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 7** (адаптация с языка R на Python)

## Краткая теория

Линейная регрессия описывает зависимость:
$$
Y' = b_0 + b_1 x_1 + ... + b_k x_k
$$

- $Y'$ — модельные (predicted) значения
- $b_0, b_1, ...$ — коэффициенты (оцениваются МНК)
- Остаток: $d = Y - Y'$

**Проверка модели:**
1. F-тест — есть ли линейная зависимость в целом
2. t-тесты коэффициентов — значимость отдельных предикторов
3. Анализ остатков:
   - нормальность (Shapiro–Wilk)
   - независимость / отсутствие автокорреляции
   - гомоскедастичность
   - малая ошибка (MSE)

В Python: **`statsmodels.formula.api.ols`** или **`sklearn.linear_model.LinearRegression`**.


## 0. Импорт библиотек


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_breuschpagan

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")
print("Библиотеки загружены")


---
## 1. Пример из задания: датасет `trees`

Прогноз Volume по Girth и Height.


In [ ]:
url_trees = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/trees.csv"
trees = pd.read_csv(url_trees, index_col=0)
print(trees.head())
print("\nРазмерность:", trees.shape)


In [ ]:
print("=== Shapiro–Wilk ===")
for col in trees.columns:
    W, p = stats.shapiro(trees[col])
    print(f"{col:8s}: W={W:.4f}, p={p:.4f}")


In [ ]:
print("Корреляции (Spearman):")
print(trees.corr(method="spearman").round(3))

sns.pairplot(trees)
plt.suptitle("Взаимосвязи в trees", y=1.02)
plt.show()


### Множественная линейная регрессия

В R: `lm(Volume ~ Girth + Height)`


In [ ]:
model = smf.ols("Volume ~ Girth + Height", data=trees).fit()
print(model.summary())


### Анализ остатков


In [ ]:
resid = model.resid

W, p_sw = stats.shapiro(resid)
print(f"Shapiro–Wilk остатков: W={W:.4f}, p={p_sw:.4f}")
print("  → нормальны" if p_sw > 0.05 else "  → НЕ нормальны")

bg = acorr_breusch_godfrey(model, nlags=1)
print(f"\nBreusch–Godfrey: LM={bg[0]:.4f}, p={bg[1]:.4f}")
print("  → нет автокорреляции" if bg[1] > 0.05 else "  → есть автокорреляция")

bp = het_breuschpagan(resid, model.model.exog)
print(f"\nBreusch–Pagan: BP={bp[0]:.4f}, p={bp[1]:.4f}")
print("  → гомоскедастичность" if bp[1] > 0.05 else "  → гетероскедастичность")

mse = np.mean(resid ** 2)
print(f"\nMSE = {mse:.4f}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].scatter(model.fittedvalues, resid, edgecolors="k", alpha=0.7)
axes[0, 0].axhline(0, color="red", ls="--")
axes[0, 0].set_xlabel("Fitted")
axes[0, 0].set_ylabel("Residuals")
axes[0, 0].set_title("Residuals vs Fitted")

sm.qqplot(resid, line="s", ax=axes[0, 1])
axes[0, 1].set_title("Normal Q-Q")

axes[1, 0].scatter(model.fittedvalues, np.sqrt(np.abs(resid)), edgecolors="k", alpha=0.7)
axes[1, 0].set_xlabel("Fitted")
axes[1, 0].set_ylabel("sqrt(|Residual|)")
axes[1, 0].set_title("Scale-Location")

axes[1, 1].plot(trees["Volume"].values, "o-", label="Observed", alpha=0.7)
axes[1, 1].plot(model.fittedvalues.values, "s-", label="Predicted", alpha=0.7)
axes[1, 1].set_xlabel("Observation")
axes[1, 1].set_ylabel("Volume")
axes[1, 1].set_title("Observed vs Predicted")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


---
## 2. Задание: данные по черепахам (Turtle)

Используем датасет с измерениями черепах:
- **weight** (Mass)
- **length** (StraightlineCL)
- **width** (MaxCW)
- **height** (ShellHeightatHinge)


In [ ]:
url_turtle = "https://raw.githubusercontent.com/JA-McLean/STOR455/master/data/Turtles.csv"
raw = pd.read_csv(url_turtle)

turtle = raw[["Mass", "StraightlineCL", "MaxCW", "ShellHeightatHinge"]].copy()
turtle.columns = ["weight", "length", "width", "height"]
turtle = turtle.dropna()

print("Размерность:", turtle.shape)
print(turtle.head())
print("\nОписательные статистики:")
print(turtle.describe().round(2))


### 2.1. Shapiro–Wilk для каждого столбца


In [ ]:
print("=== Shapiro–Wilk (Turtle) ===")
for col in turtle.columns:
    W, p = stats.shapiro(turtle[col])
    verdict = "нормальное" if p > 0.05 else "НЕ нормальное"
    print(f"{col:8s}: W={W:.4f}, p={p:.4e}  →  {verdict}")


### 2.2. Корреляционная матрица и значимые связи


In [ ]:
corr = turtle.corr(method="spearman")
print("Корреляции (Spearman):")
print(corr.round(3))

print("\nЗначимость (Spearman):")
cols = turtle.columns.tolist()
for i, c1 in enumerate(cols):
    for c2 in cols[i+1:]:
        r, p = stats.spearmanr(turtle[c1], turtle[c2])
        sig = "*" if p < 0.05 else ""
        print(f"  {c1:8s} – {c2:8s}: r={r:6.3f}, p={p:.3e} {sig}")


In [ ]:
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, square=True)
plt.title("Корреляционная матрица (Turtle, Spearman)")
plt.tight_layout()
plt.show()

sns.pairplot(turtle)
plt.suptitle("Парные зависимости", y=1.02)
plt.show()


### 2.3. Множественная регрессия: weight ~ length + width + height


In [ ]:
model_t = smf.ols("weight ~ length + width + height", data=turtle).fit()
print(model_t.summary())


### 2.4. Анализ остатков


In [ ]:
resid_t = model_t.resid

W, p_sw = stats.shapiro(resid_t)
print(f"Shapiro–Wilk остатков: W={W:.4f}, p={p_sw:.4e}")
print("  → нормальны" if p_sw > 0.05 else "  → НЕ нормальны")

bg = acorr_breusch_godfrey(model_t, nlags=1)
print(f"\nBreusch–Godfrey: LM={bg[0]:.4f}, p={bg[1]:.4e}")
print("  → нет автокорреляции" if bg[1] > 0.05 else "  → есть автокорреляция")

bp = het_breuschpagan(resid_t, model_t.model.exog)
print(f"\nBreusch–Pagan: BP={bp[0]:.4f}, p={bp[1]:.4e}")
print("  → гомоскедастичность" if bp[1] > 0.05 else "  → гетероскедастичность")

mse_t = np.mean(resid_t ** 2)
print(f"\nMSE = {mse_t:.4f}")
print(f"RMSE = {np.sqrt(mse_t):.4f}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].scatter(model_t.fittedvalues, resid_t, edgecolors="k", alpha=0.6)
axes[0, 0].axhline(0, color="red", ls="--")
axes[0, 0].set_xlabel("Fitted")
axes[0, 0].set_ylabel("Residuals")
axes[0, 0].set_title("Residuals vs Fitted")

sm.qqplot(resid_t, line="s", ax=axes[0, 1])
axes[0, 1].set_title("Normal Q-Q")

axes[1, 0].scatter(model_t.fittedvalues, np.sqrt(np.abs(resid_t)), edgecolors="k", alpha=0.6)
axes[1, 0].set_xlabel("Fitted")
axes[1, 0].set_ylabel("sqrt(|Residual|)")
axes[1, 0].set_title("Scale-Location")

axes[1, 1].scatter(turtle["weight"], model_t.fittedvalues, edgecolors="k", alpha=0.6)
mn = min(turtle["weight"].min(), model_t.fittedvalues.min())
mx = max(turtle["weight"].max(), model_t.fittedvalues.max())
axes[1, 1].plot([mn, mx], [mn, mx], "r--")
axes[1, 1].set_xlabel("Observed weight")
axes[1, 1].set_ylabel("Predicted weight")
axes[1, 1].set_title("Observed vs Predicted")

plt.tight_layout()
plt.show()


### Выводы по модели (шаблон)

1. Посмотрите **R² / Adj. R²** — какую долю дисперсии объясняет модель.
2. **F-statistic** и его p-value — значима ли модель в целом.
3. **Коэффициенты** и их p-value — какие предикторы значимы.
4. **Остатки:** нормальность, автокорреляция, гетероскедастичность, MSE.
5. Если остатки «плохие» или R² низкий — модель может быть неадекватна.

---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `lm(y ~ x1 + x2)` | `smf.ols("y ~ x1 + x2", data=df).fit()` |
| `summary(lm)` | `model.summary()` |
| `lm$residuals` | `model.resid` |
| `lm$fitted.values` | `model.fittedvalues` |
| `shapiro.test(resid)` | `stats.shapiro(resid)` |
| `bgtest(lm)` | `acorr_breusch_godfrey(model)` |
| `bptest(lm)` / White | `het_breuschpagan(resid, exog)` |
| `mean(resid^2)` | `np.mean(resid**2)` |
| `plot(lm)` | scatter + `sm.qqplot` |

---
## Рекомендации

1. Устанавливайте: `pip install statsmodels`
2. Для диагностики удобны residual plots и Q–Q plot.
3. При нарушении нормальности остатков рассмотрите преобразования (log).
4. Документация: [statsmodels OLS](https://www.statsmodels.org/stable/regression.html)

**Удачи с выполнением Задания 7!**
